# Day 3 - GroupBy, Aggregations & Date Functions

This notebook covers aggregation operations and date functions in PySpark:
- Reading Parquet files
- Basic aggregations (without GroupBy)
- GroupBy with aggregations
- Understanding shuffle operations
- Date and time functions


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder\
    .appName("spark_day3")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/06 16:37:00 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/06 16:37:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/biswa/practice/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/06 16:37:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Load Parquet Data

Read the parquet data saved in Day 1. Parquet preserves schema and is efficient for analytics.

In [5]:
parquet_path = r"../dataset/parquet_output"
df = spark.read.parquet(parquet_path)

In [6]:
from pyspark.sql import functions as F

## 3. Basic Aggregations (without GroupBy)

Aggregations without `groupBy()` compute over the entire DataFrame, returning a single row.

In [19]:
df_1 = df.select(
    F.count("ID").alias("total_customers"),
    F.min("Score").alias("min_score"),
    F.avg("Score").alias("avg_score"),
    F.max("Score").alias("max_score")
)
df_1.show()

+---------------+---------+---------+---------+
|total_customers|min_score|avg_score|max_score|
+---------------+---------+---------+---------+
|              5|      350|    625.0|      900|
+---------------+---------+---------+---------+



## 4. GroupBy with Aggregations

`groupBy()` groups rows by column(s), then `.agg()` applies aggregation functions per group.
**Triggers a Shuffle** - data is redistributed across partitions by group key.

In [17]:
df_2 = df.groupBy("Country").agg(
    F.count("ID").alias("total_customers"),
    F.min("Score").alias("min_score"),
    F.avg("Score").alias("avg_score"),
    F.max("Score").alias("max_score")
)
df_2.show()

+-------+---------------+---------+---------+---------+
|Country|total_customers|min_score|avg_score|max_score|
+-------+---------------+---------+---------+---------+
|Germany|              2|      350|    425.0|      500|
|    USA|              3|      750|    825.0|      900|
+-------+---------------+---------+---------+---------+



## 5. Understanding Shuffle Operations

### What is a Shuffle?
A shuffle is the process of redistributing data across partitions so that rows with the same key end up in the same partition. This happens during:
- `groupBy()` operations
- `join()` operations
- `repartition()` / `coalesce()`
- `orderBy()` / `sort()`

### Shuffle Phases:
1. **Map Phase** - Each partition processes its data, writes to local disk
2. **Shuffle Phase** - Data is transferred across network to target partitions
3. **Reduce Phase** - Target partitions read and aggregate data

### Performance Impact:
- **Network I/O** - Data moves between executors
- **Disk I/O** - Spill to disk if memory insufficient
- **Serialization** - Data must be serialized for transfer

### Optimization Tips:
- Filter early (before groupBy)
- Use `coalesce()` instead of `repartition()` when reducing partitions
- Consider broadcast joins for small DataFrames
- Tune `spark.sql.shuffle.partitions` (default 200)

## 6. Date and Time Functions

Spark provides a rich set of functions to parse, format, extract and manipulate dates and timestamps:
- Get the current date or timestamp
- Parse text strings into dates
- Extract components like year, month and day
- Add or subtract days and months
- Truncate timestamps down to a unit


### 6.1 Current Date and Timestamp

- `F.current_date()` returns today's date (yyyy-MM-dd)
- `F.current_timestamp()` returns the current timestamp with time
- Useful for default values and time-based filtering


In [9]:
from pyspark.sql import functions as F

date = F.current_date()
spark.range(1).select(date).show()

+--------------+
|current_date()|
+--------------+
|    2026-08-06|
+--------------+



In [ ]:
time = F.current_timestamp()
spark.range(1).select(time).show()

+--------------------+
| current_timestamp()|
+--------------------+
|2026-08-06 16:48:...|
+--------------------+



### 6.2 Parsing and Formatting Dates

- `F.to_date(col, format)`: Parses a text string and converts it to a pure date (yyyy-MM-dd)
- `F.to_timestamp(col, format)`: Parses a text string and converts it to a full timestamp with hours, minutes and seconds
- `F.date_format(col, format)`: The reverse operation - converts a Date/Timestamp back into a pretty text string (e.g., 2026-08-06 to "Thursday, August 06")


In [28]:
date_1 = "2024-12-08"
F.to_date(F.lit(date_1), "yyyy-MM-dd")

Column<'to_date('2024-12-08', 'yyyy-MM-dd')'>

In [30]:
F.date_format(F.lit(date_1), "yyyy-MM-dd")

Column<'date_format('2024-12-08', 'yyyy-MM-dd')'>

### 6.3 Extracting Date Components

Break a timestamp into its individual parts:

- `F.year(col)`: Extracts the 4-digit year (e.g., 2026)
- `F.quarter(col)`: Returns the fiscal quarter number (1 to 4)
- `F.month(col)`: Extracts the month number (1 to 12)
- `F.dayofmonth(col)`: Extracts the day of the month (1 to 31)
- `F.dayofweek(col)`: Returns the index day of the week (1 for Sunday, 7 for Saturday)
- `F.dayofyear(col)`: Returns the day number of the year (1 to 366)
- `F.weekofyear(col)`: Returns the calendar week number of the year (1 to 53)
- `F.hour(col)`: Extracts the hour block (0 to 23)
- `F.minute(col)`: Extracts the minute block (0 to 59)
- `F.second(col)`: Extracts the seconds block (0 to 59)

Read the Orders dataset which contains the `CreationTime` timestamp column:


In [49]:
path = r"../dataset/Orders.csv"
df_3 = spark.read.csv(path, header=True)
year = df_3.select(F.year(F.col('CreationTime')))
month = df_3.select(F.month(F.col('CreationTime')))

month.show()

+-------------------+
|month(CreationTime)|
+-------------------+
|                  1|
|                  1|
|                  1|
|                  1|
|                  2|
|                  2|
|                  2|
|                  2|
|                  3|
|                  3|
+-------------------+



### 6.4 Date Arithmetic

- `F.date_add(col, days)`: Adds a specific number of days forward to a date
- `F.date_sub(col, days)`: Subtracts a specific number of days backward from a date
- `F.add_months(col, months)`: Shifts a date forward or backward by a specified number of full months
- `F.datediff(end_col, start_col)`: Calculates the exact number of days between two separate date columns
- `F.months_between(end_col, start_col)`: Calculates the number of months between two dates (returns a decimal number, like 2.5 months)


In [ ]:
df_3.select(F.to_date("CreationTime"),F.date_add("CreationTime", 1).alias("next_dat")).show()

+---------------------+----------+
|to_date(CreationTime)|  next_dat|
+---------------------+----------+
|           2025-01-01|2025-01-02|
|           2025-01-05|2025-01-06|
|           2025-01-10|2025-01-11|
|           2025-01-20|2025-01-21|
|           2025-02-01|2025-02-02|
|           2025-02-06|2025-02-07|
|           2025-02-16|2025-02-17|
|           2025-02-18|2025-02-19|
|           2025-03-10|2025-03-11|
|           2025-03-16|2025-03-17|
+---------------------+----------+



### 6.5 Truncating Dates

- `F.date_trunc(format, col)`: Resets a timestamp down to the start of a specified unit (e.g., "MM" resets a timestamp to the first day of that month at 00:00:00)
- `F.last_day(col)`: Returns the final valid calendar date of the month (e.g., 2026-02-28 or 2024-02-29 for leap years)
- `F.next_day(col, "dayOfWeek")`: Finds the exact date of the next specified day of the week (e.g., F.next_day(col, "Mon") finds the upcoming Monday)


In [61]:
day_1_of_each_month = df_3.select(F.date_trunc("month","CreationTime"))
day_1_of_each_month.show()

+-------------------------------+
|date_trunc(month, CreationTime)|
+-------------------------------+
|            2025-01-01 00:00:00|
|            2025-01-01 00:00:00|
|            2025-01-01 00:00:00|
|            2025-01-01 00:00:00|
|            2025-02-01 00:00:00|
|            2025-02-01 00:00:00|
|            2025-02-01 00:00:00|
|            2025-02-01 00:00:00|
|            2025-03-01 00:00:00|
|            2025-03-01 00:00:00|
+-------------------------------+



## 7. Stop Spark Session

Always stop the session to release cluster resources.


In [62]:
spark.stop()